# Tuning a Delta table with delta-explain

A table layout is a bet on which queries matter. This notebook makes the bet
**visible**: we take real NYC taxi trips, lay them out four ways, and use
`delta-explain` to measure - before running a single query - how much file
pruning each layout actually buys.

The arc is the one every table goes through:

0. **Baseline** - an unpartitioned pile of files. Nothing prunes.
1. **Attempt 1** - partition by the wrong column. It gets *worse*.
2. **Attempt 2** - partition by the right column. Date queries fly; the rest don't.
3. **Attempt 3** - partition *and* order for skipping. Both axes prune.

Every number below is measured by `delta-explain` on a real Delta table
this notebook writes. Nothing is simulated, and `delta-explain` never reads
the parquet data - it reads only the transaction log.

**Requirements**: the `delta-explain` binary on `$PATH` (or set `DX_BIN`),
plus `deltalake` and `pyarrow`. The taxi source downloads once (~48 MB); set
`TAXI_SRC` to a local copy to skip it.

In [1]:
import json, os, shutil, subprocess, tempfile, urllib.request
from pathlib import Path
import pyarrow as pa, pyarrow.compute as pc, pyarrow.parquet as pq
from deltalake import write_deltalake

DX_BIN = os.environ.get("DX_BIN", "delta-explain")
WORK = Path(tempfile.mkdtemp(prefix="taxi-opt-"))

SRC = os.environ.get("TAXI_SRC")
if not SRC:
    SRC = str(WORK / "yellow_tripdata_2024-01.parquet")
    url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
    if not Path(SRC).exists():
        print("downloading NYC TLC yellow-taxi 2024-01 ...")
        urllib.request.urlretrieve(url, SRC)
print("delta-explain:", subprocess.run([DX_BIN, "--version"], capture_output=True, text=True).stdout.strip())

delta-explain: delta-explain 0.6.0


## The data and the queries

We keep the first week of January 2024 and a handful of real columns, then
**shuffle** the rows: a table that grows by appending as trips arrive has no
natural ordering, so every file ends up spanning the whole week and every
fare. That is the realistic starting point - and the reason nothing prunes.

We will judge every layout against three representative queries:

- **date** - `pickup_date = '2024-01-03'` (a day's trips)
- **fare** - `fare_amount > 60` (the expensive rides)
- **date + fare** - both at once

In [2]:
COLUMNS = ["tpep_pickup_datetime", "trip_distance", "PULocationID",
           "DOLocationID", "payment_type", "fare_amount", "tip_amount", "total_amount"]

t = pq.read_table(SRC, columns=COLUMNS).slice(0, 900_000)
t = t.append_column("pickup_date", pc.strftime(t["tpep_pickup_datetime"], format="%Y-%m-%d"))
week = ["2024-01-0%d" % d for d in range(1, 8)]
t = t.filter(pc.is_in(t["pickup_date"], value_set=pa.array(week)))

# Deterministic shuffle (a linear-congruential permutation of row indices):
# break time-locality so an append-order table spans the week in every file.
n = t.num_rows
order = sorted(range(n), key=lambda i: (1103515245 * i + 12345) % 2147483647)
trips = t.take(pa.array(order))
print(f"{trips.num_rows:,} trips across {len(week)} days")

QUERIES = {
    "date":      "pickup_date = '2024-01-03'",
    "fare":      "fare_amount > 60",
    "date+fare": "pickup_date = '2024-01-03' AND fare_amount > 60",
}

581,600 trips across 7 days


## The instrument

One helper writes a layout and asks `delta-explain`, per query, what fraction
of files it would eliminate. We read the machine-readable JSON (`--format
json`), so we are reading exactly the contract a CI gate would.

In [3]:
def write_layout(name, table, **kw):
    d = WORK / name
    if d.exists():
        shutil.rmtree(d)
    write_deltalake(str(d), table, mode="error", **kw)
    n_files = len(list(d.rglob("*.parquet")))
    return d, n_files

def pruning(table_dir):
    row = {}
    for label, pred in QUERIES.items():
        out = subprocess.run([DX_BIN, str(table_dir), "-w", pred, "--format", "json"],
                             capture_output=True, text=True)
        rep = json.loads(out.stdout)
        row[label] = (round(rep["total_pruning_pct"]), rep["total_files"], rep["final_files"])
    return row

def show(name, n_files, row):
    print(f"{name}   ({n_files} files)")
    for label, (pct, tot, fin) in row.items():
        bar = "#" * (pct // 5)
        print(f"  {label:10} {pct:3d}%  {tot:>3} -> {fin:<3}  {bar}")

RESULTS = {}

## Layout 0 - the baseline: an unpartitioned pile

We write the shuffled trips with no partitioning and a small target file size,
so it splits into a couple dozen files. This is the table nobody designed - it
just accreted.

In [4]:
d, nf = write_layout("L0", trips, target_file_size=500_000)
RESULTS["0. unpartitioned"] = (nf, pruning(d))
show("Layout 0: unpartitioned", nf, RESULTS["0. unpartitioned"][1])

Layout 0: unpartitioned   (23 files)
  date         0%   23 -> 23   
  fare         0%   23 -> 23   
  date+fare    0%   23 -> 23   


Let's read the full report for the date query, not just the number:

In [5]:
print(subprocess.run([DX_BIN, str(d), "-w", QUERIES["date"]], capture_output=True, text=True).stdout)

Delta table: /tmp/taxi-opt-5jn79f14/L0
Version:     0
Predicate:   pickup_date = '2024-01-03'

Predicate Analysis:
  partition-safe: -
  stats-safe:     pickup_date = '2024-01-03'
  stats coverage:
    pickup_date [min_max]: 23/23 candidate files (100%)
  unsplittable:   -
  confidence:     conservative

Files in snapshot: 23

Phase 1: Data skipping (min/max statistics) [conservative]
  predicate:       pickup_date = '2024-01-03'
  files remaining: 23  (-0, 0% pruned)



**Zero.** Every file spans the whole week, so `pickup_date` is a `stats-safe`
fragment whose min/max range covers every day - the kernel cannot rule out a
single file. Same for fare. The query reads everything. This is the pain that
sends people to "let's partition it".

## Attempt 1 - partition by the wrong column

The instinct is "partition by what I filter on". We *do* filter by pickup zone
sometimes, and `PULocationID` looks like a fine key. Watch what happens.

In [6]:
d, nf = write_layout("L1", trips, partition_by=["PULocationID"])
RESULTS["1. by PULocationID"] = (nf, pruning(d))
show("Attempt 1: partition by PULocationID", nf, RESULTS["1. by PULocationID"][1])

Attempt 1: partition by PULocationID   (240 files)
  date         3%  240 -> 232  
  fare        16%  240 -> 201  ###
  date+fare   18%  240 -> 198  ###


Two things went wrong at once. `PULocationID` has ~260 values, so we blew the
table up into **hundreds of tiny files** (the small-files problem - each one
carries overhead, and a real engine now opens far more of them). And the date
query *still* prunes almost nothing: partitioning by zone did nothing for a
filter on date. We optimized for the wrong axis and paid for it twice.

## Attempt 2 - partition by the right column

Date is what most queries filter on, and it has the right cardinality (7 days,
not 260 zones). Partition by `pickup_date`.

In [7]:
d, nf = write_layout("L2", trips, partition_by=["pickup_date"])
RESULTS["2. by pickup_date"] = (nf, pruning(d))
show("Attempt 2: partition by pickup_date", nf, RESULTS["2. by pickup_date"][1])

Attempt 2: partition by pickup_date   (7 files)
  date        86%    7 -> 1    #################
  fare         0%    7 -> 7    
  date+fare   86%    7 -> 1    #################


The date query is now **exact** - one directory, one file, everything else
skipped before any data is touched. `delta-explain` labels this `partition-safe`
with `confidence: exact`, because partition values are compared directly, no
statistics involved.

But the fare query still prunes nothing: within a day, that single file holds
every fare, so its min/max range is useless. We fixed one axis and left the
other exactly where it was.

## Attempt 3 - partition *and* order for skipping

The last move is layout inside each partition. If we sort by `fare_amount` and
write smaller files, each file within a day covers a narrow fare band, so the
min/max statistics finally become selective - data skipping can work.

In [8]:
sorted_trips = trips.sort_by([("pickup_date", "ascending"), ("fare_amount", "ascending")])
d, nf = write_layout("L3", sorted_trips, partition_by=["pickup_date"], target_file_size=400_000)
RESULTS["3. date + fare-sorted"] = (nf, pruning(d))
show("Attempt 3: date partitions + fare-sorted files", nf, RESULTS["3. date + fare-sorted"][1])
print()
print(subprocess.run([DX_BIN, str(d), "-w", QUERIES["date+fare"]], capture_output=True, text=True).stdout)

Attempt 3: date partitions + fare-sorted files   (25 files)
  date        84%   25 -> 4    ################
  fare        68%   25 -> 8    #############
  date+fare   92%   25 -> 2    ##################



Delta table: /tmp/taxi-opt-5jn79f14/L3
Version:     0
Predicate:   pickup_date = '2024-01-03' AND fare_amount > 60

Predicate Analysis:
  partition-safe: pickup_date = '2024-01-03'
  stats-safe:     fare_amount > 60
  stats coverage:
    fare_amount [min_max]: 4/4 candidate files (100%)
  unsplittable:   -
  confidence:     conservative

Files in snapshot: 25

Phase 1: Partition pruning [exact]
  predicate:       pickup_date = '2024-01-03'
  files remaining: 4  (-21, 84% pruned)

Phase 2: Data skipping (min/max statistics) [conservative]
  predicate:       fare_amount > 60
  files remaining: 2  (-2, 50% pruned)

Total reduction: 25 -> 2 files (92% pruned)



Now **both** axes prune: the date filter eliminates whole partitions
(`partition-safe`, exact), and the fare filter skips files on min/max stats
(`stats-safe`, conservative) because consecutive files hold consecutive fare
bands. The combined query is the best of both.

The cost is visible too: we went from 7 files to ~25. That is the real
trade - more files for finer skipping - and the point is that `delta-explain`
let us *see* it instead of guessing.

## The whole story, in one table

In [9]:
print(f"{'layout':24} {'files':>6} {'date':>10} {'fare':>10} {'date+fare':>12}")
print("-" * 66)
for name, (nf, row) in RESULTS.items():
    cells = "".join(f"{row[q][0]:>9}%" for q in QUERIES).replace("%", "% ", 2)
    print(f"{name:24} {nf:>6} {row['date'][0]:>9}% {row['fare'][0]:>9}% {row['date+fare'][0]:>11}%")

layout                    files       date       fare    date+fare
------------------------------------------------------------------
0. unpartitioned             23         0%         0%           0%
1. by PULocationID          240         3%        16%          18%
2. by pickup_date             7        86%         0%          86%
3. date + fare-sorted        25        84%        68%          92%


## The tool predicts the next move

We did not reason our way from Attempt 2 to Attempt 3 - `--explain-why` did.
Ask it, on the date-partitioned layout, why the fare query still scans every
file:

In [10]:
import subprocess
print(subprocess.run([DX_BIN, str(WORK / "L2"), "-w", "fare_amount > 60", "--explain-why"],
                     capture_output=True, text=True).stdout)
shutil.rmtree(WORK)

Delta table: /tmp/taxi-opt-5jn79f14/L2
Version:     0
Predicate:   fare_amount > 60

Predicate Analysis:
  partition-safe: -
  stats-safe:     fare_amount > 60
  stats coverage:
    fare_amount [min_max]: 7/7 candidate files (100%)
  unsplittable:   -
  confidence:     conservative

Files in snapshot: 7

Phase 1: Data skipping (min/max statistics) [conservative]
  predicate:       fare_amount > 60
  files remaining: 7  (-0, 0% pruned)

Why:
  [NO_PARTITION_FILTER] The table is partitioned by pickup_date, but the predicate filters on none of those columns, so partition pruning cannot run.
    -> Filter on a partition column (pickup_date) to eliminate whole directories before data skipping.
  [WEAK_DATA_SKIPPING] Data skipping eliminated no files for 'fare_amount > 60': the per-file min/max ranges all overlap the predicate's bound.
    -> Ranges this wide usually mean the data is not sorted or clustered by that column; ordering by it so each file covers a narrower range may enable skippi

## What delta-explain gave us

We never ran a query engine, never read a parquet file, and never guessed. At
each step the tool answered the only question that matters when you tune a
layout: *for the queries I care about, how many files does this eliminate?*

- The **baseline** told us the pile was hopeless (0% everywhere).
- **Attempt 1** caught a common mistake - partitioning by a high-cardinality
  column - before it shipped: worse file count, no benefit.
- **Attempt 2** proved the right partition column, and showed exactly which
  queries it did *not* help.
- **Attempt 3** confirmed the skipping win and made its file-count cost
  explicit.

The same measurement runs in CI: `delta-explain <table> -w "<query>"
--min-pruning 80` fails the build if a change quietly drops the layout back
toward the baseline. That is the difference between a table that is fast
because someone measured, and one that is fast until it silently isn't.